# 03. Spatio-Temporal Incident Correlation & Lag Anomaly Detection

**Objective:** Residualize daily sales against trend and day-of-week seasonality, join geo-fenced Haversine incident exposure (`t`, `t+1`, `t+2`), and validate identification via `t-1` placebo falsification, category stratification, and intraday hourly alignment.

**Mode Contract:** Uses `data/processed/incidents_raw.csv` in `raw` mode (generated by Notebook `02`) and never touches `data/sample/incidents.csv` in `raw` mode.

## 1. Environment Setup & Correlation Imports

**Objective:** Configure PID-isolated matplotlib environment, rendering helpers, `StoreConfig`, and spatial lag estimators.

**Inputs / Assumptions:** Emits coordinate notice when `cfg.coords_verified == False`.

**How to Read the Output:** Confirms active mode and storefront coordinate verification status.

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path(os.environ.get("BGA_REPO_ROOT", Path.cwd())).resolve()
if not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd

from src.config import load_config
from src.notebook_utils import (
    audit_intraday_incident_hours,
    configure_mpl_env,
    date_overlap_audit,
    load_incidents_for_mode,
    placebo_lead_test,
    regime_and_trend_audit,
    render_table,
    show_fig,
    stratify_residuals_by_category,
)
from src.spatial_models import estimate_lag_effects, join_sales_with_incidents

plt = configure_mpl_env()
cfg = load_config(repo_root=REPO_ROOT)
DATA_MODE = cfg.data_mode
if not cfg.coords_verified:
    print(f"NOTICE: Using approximate Silao coords ({cfg.lat}, {cfg.lon}) with coords_verified=False.")

## 2. Mode-Aware Data Load, Date-Overlap Audit & Distributed Lag OLS

**Objective:** Load sales (`daily_df`, `hourly_df`) and mode-isolated OSINT incidents, audit calendar overlap, and estimate joint distributed lag impacts (`t`, `t+1`, `t+2`) on sales residuals.

**Inputs / Assumptions:** Exposure decays exponentially (`half_life_m=1500.0`) within `max(cfg.geofence_radii_m)` (`5000 m`).

**How to Read the Output:** Confirm `overlap_tbl` reports `status == OK` (`incident_days_in_window > 0`), then inspect `coef` and `p_value` across `t`, `t+1`, and `t+2`.

In [ ]:
if DATA_MODE == "raw":
    from src.db_loader import load_daily_sales, load_hourly_sales
    if not cfg.master_sales_db_path.is_file():
        raise FileNotFoundError(f"Missing raw SQLite DB: {cfg.master_sales_db_path}")
    daily_df = load_daily_sales(config=cfg)
    hourly_df = load_hourly_sales(config=cfg, complete_grid=True)
else:
    daily_df = pd.read_csv(cfg.sample_dir / "daily_sales.csv", parse_dates=["date"])
    hourly_df = pd.read_csv(cfg.sample_dir / "hourly_sales.csv", parse_dates=["date"])

incidents_df, incidents_source = load_incidents_for_mode(cfg, DATA_MODE, relevant_only=True)
print(f"Loaded {len(incidents_df)} incidents from {incidents_source}")

overlap_tbl = date_overlap_audit(daily_df, incidents_df)
render_table(overlap_tbl, title="Sales vs. OSINT Incident Date-Overlap Audit")
if int(overlap_tbl.loc[0, "incident_days_in_window"]) == 0:
    print("WARNING: Zero incident dates overlap the sales observation window; lag exposure vectors are all zero.")

joined = join_sales_with_incidents(
    sales_df=daily_df,
    incidents_df=incidents_df,
    store_lat=cfg.lat,
    store_lon=cfg.lon,
    radii_m=cfg.geofence_radii_m,
    max_distance_m=float(max(cfg.geofence_radii_m)),
    lags=(0, 1, 2),
    regime_dates=[cfg.tablet_channel_start_date],
    include_quincena=True,
    holiday_dates="mx",
)
joined["is_post_tablet_change"] = (pd.to_datetime(joined["date"]) >= pd.Timestamp(cfg.tablet_channel_start_date)).astype(float)
lag_results = estimate_lag_effects(
    joined,
    outcome_col="residual_pct",
    lags=(0, 1, 2),
    control_cols=("is_post_tablet_change", "is_quincena", "is_holiday"),
)
render_table(
    lag_results,
    title="Daily Distributed Lag Regression (t, t+1, t+2; Controlled for POS Shift, Quincena & MX Holidays)",
    fmt={"coef": "{:.4f}", "std_err": "{:.4f}", "t_stat": "{:.2f}", "p_value": "{:.4f}", "ci_low": "{:.4f}", "ci_high": "{:.4f}", "r_squared": "{:.4f}"},
)

## 3. Falsification, Category Stratification & Intraday Hourly Alignment

**Objective:** Stress-test identification with a `t-1` placebo lead check, stratify daily residuals by incident category, and compare intraday `(h, h+1)` affected hours against same-hour baseline revenue.

**Inputs / Assumptions:** Placebo lead CI excluding zero (`flag_pretrend == True`) warns of pre-trend or confounding bias.

**How to Read the Output:** Expect `t-1` placebo CI to straddle zero, negative mean residuals on severe disruption categories, and negative `diff_pct` in incident-hour windows.

In [ ]:
placebo_tbl = placebo_lead_test(joined, outcome_col="residual_pct")
render_table(
    placebo_tbl,
    title="Placebo Lead Falsification Check (t-1)",
    fmt={"coef": "{:.4f}", "std_err": "{:.4f}", "t_stat": "{:.2f}", "p_value": "{:.4f}", "ci_low": "{:.4f}", "ci_high": "{:.4f}"},
)
if bool(placebo_tbl.loc[0, "flag_pretrend"]):
    print("WARNING: t-1 placebo 95% CI excludes zero; investigate pre-trend confounding.")

regime_audit = regime_and_trend_audit(joined, tablet_start_date=cfg.tablet_channel_start_date)
render_table(
    regime_audit["regime_summary"],
    title="Regime Shift vs Acute Shock Audit (Pre vs Post POS Change)",
    fmt={"mean_revenue": "{:,.2f}", "mean_residual_pct": "{:.2%}", "incident_share": "{:.1%}"},
)
render_table(
    regime_audit["weather_vs_street"],
    title="Weather Confounding vs Street Incident Impact",
    fmt={"mean_residual_pct": "{:.2%}", "std_residual_pct": "{:.2%}"},
)

cat_tbl = stratify_residuals_by_category(joined, incidents_df, outcome_col="residual_pct")
render_table(
    cat_tbl,
    title="Category-Stratified Daily Sales Residuals",
    fmt={"mean_residual_pct": "{:.2%}", "std_residual_pct": "{:.2%}"},
)

intraday_tbl = audit_intraday_incident_hours(hourly_df, incidents_df, window_hours=(0, 1))
intraday_view = intraday_tbl[intraday_tbl["affected_obs"] > 0]
render_table(
    intraday_view if not intraday_view.empty else intraday_tbl,
    title="Intraday Hourly Window Audit (h, h+1 vs Same-Hour Baseline)",
    fmt={"baseline_mean_volume": "{:,.2f}", "affected_mean_volume": "{:,.2f}", "diff_pct": "{:.1%}"},
)

## 4. 2x2 Correlation & Diagnostic Visual Summary

**Objective:** Visualize daily residual shocks, distributed lag estimates with `t-1` placebo, category-level residuals, and intraday hour alignment.

**Inputs / Assumptions:** Error bars reflect 95% confidence intervals.

**How to Read the Output:** Compare `t-1` placebo against `t`/`t+1`/`t+2` and inspect whether incident categories and hours drive negative anomalies.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))

axes[0, 0].plot(joined["date"], joined["residual_pct"] * 100.0, alpha=0.65, label="Sales Residual (%)")
inc_hits = joined[joined["has_incident_lag0"]]
axes[0, 0].scatter(inc_hits["date"], inc_hits["residual_pct"] * 100.0, color="#d95f02", zorder=4, label="Incident Day (t=0)")
axes[0, 0].axhline(0.0, color="black", linestyle="--", linewidth=0.8)
axes[0, 0].set_title("Daily Revenue Residuals vs OSINT Incident Days")
axes[0, 0].set_ylabel("Residual (%)")
axes[0, 0].legend()

lag_plot = pd.concat([
    placebo_tbl[["lag", "coef", "ci_low", "ci_high"]],
    lag_results[["lag", "coef", "ci_low", "ci_high"]],
], ignore_index=True).sort_values("lag")
yerr = (lag_plot["ci_high"] - lag_plot["coef"]) * 100.0
axes[0, 1].errorbar(lag_plot["lag"], lag_plot["coef"] * 100.0, yerr=yerr, fmt="o-", capsize=4, color="#1b9e77")
axes[0, 1].axhline(0.0, color="black", linestyle="--", linewidth=0.8)
axes[0, 1].axvline(-0.5, color="gray", linestyle=":", linewidth=0.9)
axes[0, 1].set_xticks([-1, 0, 1, 2])
axes[0, 1].set_xticklabels(["t-1 (Placebo)", "t (Same Day)", "t+1", "t+2"])
axes[0, 1].set_title("Distributed Lag Exposure Effect (%) with 95% CI")
axes[0, 1].set_ylabel("Effect (%)")

cat_colors = ["#999999" if c == "no_incident" else "#2b8cbe" for c in cat_tbl["category"]]
bars = axes[1, 0].bar(cat_tbl["category"], cat_tbl["mean_residual_pct"] * 100.0, color=cat_colors)
for bar, n_val in zip(bars, cat_tbl["n_days"]):
    y_pos = bar.get_height()
    va = "bottom" if y_pos >= 0 else "top"
    axes[1, 0].text(bar.get_x() + bar.get_width() / 2.0, y_pos, f"n={int(n_val)}", ha="center", va=va, fontsize=8.5)
axes[1, 0].axhline(0.0, color="black", linestyle="--", linewidth=0.8)
axes[1, 0].set_title("Mean Daily Residual (%) by Incident Category")
axes[1, 0].set_ylabel("Mean Residual (%)")
axes[1, 0].tick_params(axis="x", rotation=15)

axes[1, 1].plot(intraday_tbl["hour_int"], intraday_tbl["baseline_mean_volume"], marker="o", label="Baseline Same-Hour Mean", color="#3288bd")
aff_pts = intraday_tbl.dropna(subset=["affected_mean_volume"])
if not aff_pts.empty:
    axes[1, 1].scatter(aff_pts["hour_int"], aff_pts["affected_mean_volume"], color="#d53e4f", s=55, zorder=4, label="Affected Hours (h, h+1)")
axes[1, 1].set_title("Intraday Affected-Hour vs Baseline Hourly Volume")
axes[1, 1].set_xlabel("Hour of Day")
axes[1, 1].set_ylabel(f"Hourly Revenue ({cfg.currency})")
axes[1, 1].legend()

show_fig(fig)